# ConfigStream Wiki

Welcome to the **ConfigStream** documentation. ConfigStream is an automated, high-performance VPN configuration aggregator designed to provide reliable, free access to the open internet.

## Core Concepts

*   **Zero Budget Architecture:** The system runs entirely on free, resilient cloud infrastructure (GitHub Actions, Pages, Cloudflare), ensuring long-term sustainability and censorship resistance.
*   **Hybrid Engine:** We combine the flexibility of **Python** for data processing and intelligence with the raw performance of **Go** for massive-scale network testing.
*   **Smart Routing ("The Sniper"):** Our configurations are optimized to route traffic intelligently, bypassing restrictions while maintaining speed.
*   **Proxy Washing:** blocked IPs are automatically "washed" through Cloudflare WARP to restore connectivity.

## Getting Started

1.  **Download:** Visit the [Home Page](../index.html) to get the latest subscription links.
2.  **Import:** Use a compatible client like **V2RayNG**, **Shadowrocket**, **Streisand**, or **Hiddify**.
3.  **Connect:** Select a proxy and connect.

## Documentation Index

*   **[Architecture](Architecture_v2.md):** A deep dive into how ConfigStream works under the hood.
*   **[Troubleshooting](Troubleshooting.md):** Solutions for common connection issues.
*   **[Contributing](09-contributing.md):** How developers can help improve the project.
*   **[API Reference](08-api-reference.md):** Details on the data structures and endpoints.

## Legal & Security

These configurations are aggregated from public sources. While we perform rigorous automated security checks (filtering malware, honeypots, and invalid certs), usage is at your own risk. We recommend avoiding sensitive transactions (banking) over public proxies.




---



# 06. Frontend & User Experience

The ConfigStream frontend is a **Progressive Web App (PWA)** designed for speed, resilience, and visualization. It adheres to a "No Build Step" philosophy for the frontend code itself (Vanilla JS), though the backend generates the data it consumes.

## Architecture

*   **Framework**: Vanilla ES6+ JavaScript. No React, Vue, or Angular.
*   **Styling**: Custom CSS variables for theming.
*   **Data Source**: Static JSON files (`metadata.json`, `proxies.json`) fetched from the `output/` directory.

### The "Cache-First" Strategy (Service Worker)

We use a custom Service Worker (`service-worker.js`) to make the site censorship-resistant.

1.  **Assets (HTML/CSS/JS)**: Cached immediately on first load. The app works offline.
2.  **Data (JSON)**: Network-First. We try to fetch the latest proxy list. If the network fails (blocked), we serve the last cached list.
3.  **Updates**: The SW checks for a new version of the app in the background.

## Security Hardening Checklist

When making changes to the frontend, you **must** adhere to these security practices to prevent XSS (Cross-Site Scripting) and other client-side attacks.

### 1. No Unsafe `innerHTML`
*   **Never** assign user-controlled data directly to `innerHTML`.
*   **Use** `textContent` for text updates.
*   **Use** `updateElement(selector, content, { method: 'textContent' })` helper.
*   **If you MUST use HTML**:
    *   Sanitize it first using `DOMPurify.sanitize()`.
    *   Use `updateElement(selector, content, { method: 'innerHTML' })`, which handles sanitization automatically (unless `trustedHTML: true` is set).

### 2. DOMPurify
*   We load `DOMPurify` (vendored in `assets/js/lib/purify.min.js`).
*   Ensure it is included in your HTML file before your scripts run.
*   Any large block of HTML constructed from data (e.g., Markdown rendering, Proxy Tables) must be passed through `DOMPurify.sanitize()`.

### 3. URL Handling
*   Validate all URLs before setting them as `href` or `src`.
*   Use `validateURL()` helper from `utils.js`.
*   Avoid `javascript:` URIs.

### 4. Dependencies
*   **Vendor everything**. Do not rely on external CDNs (they can be blocked or compromised).
*   Keep `assets/js/lib/` clean. Only minimal, audited libraries.

### 5. Content Security Policy (CSP)
*   The `index.html` should enforce a strict CSP (via meta tag or headers).
*   `script-src 'self' 'unsafe-inline' 'unsafe-eval' blob:;` (Adjusted for WASM/Inline scripts requirements, tighten where possible).

## Visualization Components

### 1. The Globe (`globe.gl`)
A WebGL-based 3D globe visualization.
*   **Data**: Latency distribution from `metadata.json`.
*   **Arcs**: Draws arcs from the user's estimated location to the proxy location.
*   **Color Coding**: Green (Fast), Yellow (Medium), Red (Slow).

### 2. Analytics (`Chart.js`)
*   **Protocol Distribution**: Doughnut chart.
*   **Country Distribution**: Bar chart.
*   **Latency Heatmap**: A scatter plot of Ping vs. Time.

### 3. Virtual Scrolling (The Proxy Table)
Rendering 5,000 DOM elements (table rows) kills the browser.
*   **Solution**: We implement **Virtual Scrolling**.
*   **Mechanism**: We only render the ~20 rows currently visible in the viewport. As the user scrolls, we dynamically recycle and update these DOM nodes.
*   **Result**: 60fps scrolling even with 100,000 proxies.

## WASM Edge Testing

This is the "Hero" feature. We allow users to verify proxies *from their own network*.

### The Bridge (`wasm_loader.js`)
1.  **Load**: Fetches `tester.wasm` (compiled from Go).
2.  **Instantiate**: Loads the WASM module into memory.
3.  **Expose**: Maps Go functions (e.g., `TestProxy`) to JavaScript functions (`window.testProxy`).

### The Challenge: Browser Sandbox
Browsers cannot open TCP sockets.
*   **WebSocket Proxies**: The WASM module uses the browser's `WebSocket` API to test `vmess+ws`, `vless+ws`, etc. These tests are **Real**.
*   **TCP Proxies**: The WASM module currently simulates a check or uses a fallback HTTP ping (if CORS allows).
*   **Future**: We are exploring WebTransport and Relay nodes.

## Vector Search (Natural Language)

*   **Input**: "Fast reliable US proxy"
*   **Process (Current Implementation)**:
    1.  Frontend tokenizes the query into lowercase keywords.
    2.  Fetches `vectors.json` (pre-computed) and proxy metadata.
    3.  Computes a simple keyword‑based relevance score on the client (protocol, country code, city, tags).
    4.  Ranks results by this score.
*   **Planned Enhancements (Not Yet Implemented)**:
    *   Replace the heuristic score with a proper vector‑space similarity measure (for example, cosine similarity over dense vectors).
    *   Move heavy ranking into a Web Worker to avoid blocking the UI thread for very large datasets.

## Time-Travel Sparklines

In the proxy table, the "Latency" column is not just a number. It is a story.
*   **Data**: `history: [100, 120, 900, 110, 115]`
*   **Visual**: A tiny SVG sparkline.
*   **Insight**: A user sees a spike (900ms) and knows the proxy is unstable, even if it says "115ms" right now.

## Internationalization (i18n)

We support RTL (Right-to-Left) languages natively for our Persian and Arabic users.
*   **Dictionary**: `assets/js/i18n.js` contains mappings.
*   **Detection**: Auto-detects browser language.
*   **Switching**: Dynamic, no reload required.




---



# Analytics Page Documentation

The **Analytics** page is the transparency engine of ConfigStream. It transforms raw data into actionable insights, allowing users and developers to understand the dynamics of the proxy ecosystem.

## Design Philosophy: "Trust Through Data"

We believe that open-source tools must be transparent. By exposing the internal metrics of our pipeline—success rates, latency distributions, and source reliability—we prove the system's efficacy.

### Key Visualizations

#### 1. Latency Heatmap (The World Map)
*   **Library**: `Chart.js` (Choropleth extension) or Custom SVG Map.
*   **Data**: Average latency per country code (ISO 3166-1 alpha-2).
*   **Insight**: Users can instantly see which regions offer the fastest connection speeds.
*   **Color Scale**: Green (Low Latency) -> Red (High Latency) -> Grey (No Data).

#### 2. Protocol Breakdown
*   **Type**: Doughnut / Pie Chart.
*   **Data**: Count of unique proxies per protocol (VLESS, VMess, Trojan, Shadowsocks, etc.).
*   **Insight**: Shows the diversity of the network. A healthy network has a balanced mix, preventing total blockage if one protocol is targeted by censors.

#### 3. Latency Distribution (The Bell Curve)
*   **Type**: Histogram.
*   **Buckets**: <500ms, 500-1000ms, 1000-2000ms, >2000ms.
*   **Insight**: Demonstrates the quality of the "Refining" process. A left-skewed graph (towards lower latency) indicates a high-quality batch.

#### 4. Top ISPs (Internet Service Providers)
*   **Type**: Horizontal Bar Chart.
*   **Data**: Top 10 ASNs (Autonomous Systems) hosting the proxies.
*   **Insight**: Reveals infrastructure trends (e.g., "Are most proxies on DigitalOcean or Cloudflare?").

### Historical Trends (Future Roadmap)
*   We plan to add a time-series graph showing the total proxy count over the last 30 days to visualize network stability and growth.

## Technical Implementation

*   **Data Source**: `metadata.json` (specifically the `stats` and `latency_distribution` fields).
*   **Responsiveness**: Charts automatically resize for mobile/desktop screens.
*   **Interaction**: Tooltips provide exact counts and percentages on hover.




---



# 10. Troubleshooting & FAQ

## Common Issues

### 1. "Configuration Import Failed"
If your client fails to import the configuration:
*   **Check the Format**: Ensure you are using the correct format for your client (e.g., `.yaml` for Clash, `.json` for Sing-box).
*   **Base64 Decoding**: Some older clients expect raw URI lists. Try decoding the Base64 string manually if your client doesn't support subscription links.
*   **Update Client**: We use modern protocols (VLESS-Reality, Hysteria2). Ensure your client is up to date (e.g., v2rayNG >= 1.8.5, Sing-box >= 1.8).

### 2. "Connected but No Internet"
*   **Time Sync**: VLESS/VMess protocols require your device time to be accurate within 90 seconds. Sync your clock.
*   **Geo-Blocking**: The proxy might be blocked by the destination site. Try a different country.
*   **ISP Blocking**: Your ISP might be blocking the specific port or protocol. Try a "Washed" proxy or a different protocol (e.g., switch from VLESS to Hysteria).

### 3. "High Latency"
*   **Real vs. Handshake**: The latency shown in the app is often just the TCP handshake time to the proxy server, not the real download speed.
*   **Route Optimization**: Use the "Auto" or "UrlTest" group in your client to automatically select the fastest node.

## Client-Specific Guides

### Android
*   **v2rayNG**: Recommended. Supports all protocols.
    1.  Copy the "Universal Subscription" link.
    2.  Open v2rayNG -> Menu -> Subscription Group Setup -> Add.
    3.  Paste link -> Update Subscription.
*   **NekoBox**: Best for Sing-box configs.
*   **Clash Meta**: Required for our Clash configs (standard Clash doesn't support VLESS).

### iOS
*   **Shadowrocket**: Paid, but best. Supports everything.
    *   Import using the "Shadowrocket" specific link for optimized tags.
*   **Streisand**: Good free alternative.
*   **Sing-box**: Official app available on TestFlight/AppStore.

### Windows / macOS
*   **v2rayN (Windows)**: The gold standard.
*   **Clash Verge (Windows/Mac)**: Modern Clash client.
*   **Sing-box (CLI/GUI)**: For advanced users.

## Advanced Usage

### How to use "The Sniper" (Router Mode)
The `singbox.json` output is designed as a "Sniper". It uses a `tun` interface but only routes traffic that matches specific rules (e.g., blocked domains).
1.  Download `singbox.json`.
2.  Run `sing-box run -c singbox.json`.
3.  Set your device gateway to the machine running Sing-box.

### How to use "The Tank" (VPN Mode)
The `singbox-vpn.json` is a "Tank". It routes **everything** through the proxy.
*   WARNING: This will route local traffic too if not configured correctly.
*   Use this when you are on a very hostile network (e.g., public WiFi) and want full encryption.

## Getting Help
If you encounter persistent issues, please open an issue on GitHub with:
1.  Your client name and version.
2.  The specific error message.
3.  Which subscription link you are using.

